In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(r'D:\Projects\Machine Learning\Supervised\Regression\medical_cost_prediction\datasets\insurance.csv')
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [4]:
df.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

## Feature Engineering

In [5]:
df['sex'].unique()

array(['female', 'male'], dtype=object)

In [6]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [7]:
## converting gender into encoded format.
df['sex'] = df['sex'].apply(lambda x: 0 if x == 'male' else 1)
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,1,27.900,0,yes,southwest,16884.92400
1,18,0,33.770,1,no,southeast,1725.55230
2,28,0,33.000,3,no,southeast,4449.46200
3,33,0,22.705,0,no,northwest,21984.47061
4,32,0,28.880,0,no,northwest,3866.85520


In [8]:
df['smoker'].unique()

array(['yes', 'no'], dtype=object)

In [9]:
## converting smokers into encoded format
df['smoker'] = df['smoker'].apply(lambda x: 0 if x == 'no' else 1)
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,1,27.900,0,1,southwest,16884.92400
1,18,0,33.770,1,0,southeast,1725.55230
2,28,0,33.000,3,0,southeast,4449.46200
3,33,0,22.705,0,0,northwest,21984.47061
4,32,0,28.880,0,0,northwest,3866.85520


In [10]:
pd.get_dummies(df, prefix=['region'], columns=['region'])

,age,sex,bmi,children,smoker,charges,region_northeast,region_northwest,region_southeast,region_southwest
0,19,1,27.900,0,1,16884.92400,False,False,False,True
1,18,0,33.770,1,0,1725.55230,False,False,True,False
2,28,0,33.000,3,0,4449.46200,False,False,True,False
3,33,0,22.705,0,0,21984.47061,False,True,False,False
4,32,0,28.880,0,0,3866.85520,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...
1333,50,0,30.970,3,0,10600.54830,False,True,False,False
1334,18,1,31.920,0,0,2205.98080,True,False,False,False
1335,18,1,36.850,0,0,1629.83350,False,False,True,False
1336,21,1,25.800,0,0,2007.94500,False,False,False,True


#### One-Hot encoding to regions

000 -> southwest

001 -> southeast

010 -> northwest

100 -> northeast


In [11]:
df_encoded = pd.get_dummies(df, prefix=['region'], columns=['region'], dtype=int)

## df_encoded = df_encoded.str.replace('region_', '', regex = False)
df_encoded.columns = df_encoded.columns.str.removeprefix('region_')
df_encoded.drop('southwest', axis = 1, inplace = True)
df_encoded


,age,sex,bmi,children,smoker,charges,northeast,northwest,southeast
0,19,1,27.900,0,1,16884.92400,0,0,0
1,18,0,33.770,1,0,1725.55230,0,0,1
2,28,0,33.000,3,0,4449.46200,0,0,1
3,33,0,22.705,0,0,21984.47061,0,1,0
4,32,0,28.880,0,0,3866.85520,0,1,0
...,...,...,...,...,...,...,...,...,...
1333,50,0,30.970,3,0,10600.54830,0,1,0
1334,18,1,31.920,0,0,2205.98080,1,0,0
1335,18,1,36.850,0,0,1629.83350,0,0,1
1336,21,1,25.800,0,0,2007.94500,0,0,0


In [12]:
df_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   age        1338 non-null   int64  
 1   sex        1338 non-null   int64  
 2   bmi        1338 non-null   float64
 3   children   1338 non-null   int64  
 4   smoker     1338 non-null   int64  
 5   charges    1338 non-null   float64
 6   northeast  1338 non-null   int64  
 7   northwest  1338 non-null   int64  
 8   southeast  1338 non-null   int64  
dtypes: float64(2), int64(7)
memory usage: 94.2 KB


## Train, Test and Split

In [13]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop('charges', axis = 1)
y = df_encoded['charges']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
X_train.shape, X_test.shape

((1003, 8), (335, 8))

## Linear Regression

In [14]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [15]:
y_pred = lin_reg.predict(X_test)

In [16]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mse)
score = r2_score(y_test, y_pred)
adjusted_score = 1 - (1-score)*(len(y_test) - 1) / (len(y_test) - X_test.shape[1] -1)

print(f"Mean Squared Error: {mse}")
print(f"Mean Absolute Error: {mae}")
print(f"Root Mean Squared Error: {rmse}")
print(f"R2 Score: {score}")
print(f"Adjusted R2 Score: {adjusted_score}")

Mean Squared Error: 35117755.73613631
Mean Absolute Error: 4243.654116653136
Root Mean Squared Error: 5926.023602394468
R2 Score: 0.7672642952734358
Adjusted R2 Score: 0.7615529896359741


## Polynomial Regression

In [17]:
from sklearn.preprocessing import PolynomialFeatures

poly_reg_lst = []
for i in range(2, 6):
    poly = PolynomialFeatures(degree = i, include_bias = True)
    
    X_train_poly = poly.fit_transform(X_train)
    X_test_poly = poly.transform(X_test)
    
    reg = LinearRegression()
    reg.fit(X_train_poly, y_train)
    poly_reg_lst.append({
        'degree': i,
        'model': reg,
        'x_train' : X_train_poly,
        'x_test' : X_test_poly,
        'poly' : poly
    })

In [18]:
for model in poly_reg_lst:
    reg = model['model']
    y_pred = reg.predict(model['x_test'])
    
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    score = r2_score(y_test, y_pred)
    adjusted_score = 1 - (1-score)*(len(y_test) - 1) / (len(y_test) - model['x_test'].shape[1] -1)

    print(f"Degree of the Polynomial: {model['degree']}")
    print(f"Mean Squared Error: {mse}")
    print(f"Mean Absolute Error: {mae}")
    print(f"Root Mean Squared Error: {rmse}")
    print(f"R2 Score: {score}")
    print(f"Adjusted R2 Score: {adjusted_score}")
    print("\n ---------------------------------------------------------------\n")
    
    

Degree of the Polynomial: 2
Mean Squared Error: 21990968.110006064
Mean Absolute Error: 2781.7101811018606
Root Mean Squared Error: 4689.452858277398
R2 Score: 0.8542593809479934
Adjusted R2 Score: 0.8315662049710373

 ---------------------------------------------------------------

Degree of the Polynomial: 3
Mean Squared Error: 24229055.018639643
Mean Absolute Error: 2965.0862000061325
Root Mean Squared Error: 4922.301800848831
R2 Score: 0.8394269201884311
Adjusted R2 Score: 0.6826543866445917

 ---------------------------------------------------------------

Degree of the Polynomial: 4
Mean Squared Error: 42602570.54182125
Mean Absolute Error: 3731.8381638758606
Root Mean Squared Error: 6527.0644658852
R2 Score: 0.7176602242833183
Adjusted R2 Score: 1.5857235098718738

 ---------------------------------------------------------------

Degree of the Polynomial: 5
Mean Squared Error: 168862648.68294764
Mean Absolute Error: 5558.145572530246
Root Mean Squared Error: 12994.716183239541
R

## Lasso Regression

In [19]:
from sklearn.linear_model import Lasso 

lasso_reg = Lasso()
lasso_reg.fit(X_train, y_train)

y_pred = lasso_reg.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mse)
score = r2_score(y_test, y_pred)
adjusted_score = 1 - (1-score)*(len(y_test) - 1) / (len(y_test) - X_test.shape[1] -1)

print(f"Mean Squared Error: {mse}")
print(f"Mean Absolute Error: {mae}")
print(f"Root Mean Squared Error: {rmse}")
print(f"R2 Score: {score}")
print(f"Adjusted R2 Score: {adjusted_score}")

Mean Squared Error: 35118565.45195941
Mean Absolute Error: 4243.717413071323
Root Mean Squared Error: 5926.091920647148
R2 Score: 0.7672589290483234
Adjusted R2 Score: 0.7615474917243559


## Ridge Regression

In [20]:
from sklearn.linear_model import Ridge

ridge_reg = Ridge()

ridge_reg.fit(X_train, y_train)
y_pred = ridge_reg.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mse)
score = r2_score(y_test, y_pred)
adjusted_score = 1 - (1-score)*(len(y_test) - 1) / (len(y_test) - X_test.shape[1] -1)

print(f"Mean Squared Error: {mse}")
print(f"Mean Absolute Error: {mae}")
print(f"Root Mean Squared Error: {rmse}")
print(f"R2 Score: {score}")
print(f"Adjusted R2 Score: {adjusted_score}")

Mean Squared Error: 35160344.253732815
Mean Absolute Error: 4256.543520877248
Root Mean Squared Error: 5929.615860553938
R2 Score: 0.7669820486307244
Adjusted R2 Score: 0.7612638166952821


In [21]:
from sklearn.linear_model import ElasticNet

elastic_net_reg = ElasticNet()
elastic_net_reg.fit(X_train, y_train)

y_pred = elastic_net_reg.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mse)
score = r2_score(y_test, y_pred)
adjusted_score = 1 - (1-score)*(len(y_test) - 1) / (len(y_test) - X_test.shape[1] -1)

print(f"Mean Squared Error: {mse}")
print(f"Mean Absolute Error: {mae}")
print(f"Root Mean Squared Error: {rmse}")
print(f"R2 Score: {score}")
print(f"Adjusted R2 Score: {adjusted_score}")

Mean Squared Error: 90845289.87547068
Mean Absolute Error: 7446.311458257479
Root Mean Squared Error: 9531.279550798554
R2 Score: 0.3979415222567736
Adjusted R2 Score: 0.3831670810851606


## Lasso CV

In [31]:
from sklearn.linear_model import LassoCV

lasso_cv_reg = LassoCV(cv=10) ## By default, cv is 5
lasso_cv_reg.fit(X_train, y_train)

y_pred = lasso_cv_reg.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mse)
score = r2_score(y_test, y_pred)
adjusted_score = 1 - (1-score)*(len(y_test) - 1) / (len(y_test) - X_test.shape[1] -1)

print(f"Mean Squared Error: {mse}")
print(f"Mean Absolute Error: {mae}")
print(f"Root Mean Squared Error: {rmse}")
print(f"R2 Score: {score}")
print(f"Adjusted R2 Score: {adjusted_score}")


Mean Squared Error: 35442376.388021015
Mean Absolute Error: 4291.755202734245
Root Mean Squared Error: 5953.350013901502
R2 Score: 0.7651129386562063
Adjusted R2 Score: 0.7593488389913279


In [32]:
from sklearn.linear_model import RidgeCV

ridge_cv_reg = RidgeCV(cv=10) ## By default, cv is 5
ridge_cv_reg.fit(X_train, y_train)

y_pred = elastic_net_reg.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mse)
score = r2_score(y_test, y_pred)
adjusted_score = 1 - (1-score)*(len(y_test) - 1) / (len(y_test) - X_test.shape[1] -1)

print(f"Mean Squared Error: {mse}")
print(f"Mean Absolute Error: {mae}")
print(f"Root Mean Squared Error: {rmse}")
print(f"R2 Score: {score}")
print(f"Adjusted R2 Score: {adjusted_score}")

Mean Squared Error: 90845289.87547068
Mean Absolute Error: 7446.311458257479
Root Mean Squared Error: 9531.279550798554
R2 Score: 0.3979415222567736
Adjusted R2 Score: 0.3831670810851606


In [33]:
from sklearn.linear_model import ElasticNetCV

elastic_net_cv_reg = ElasticNetCV(cv=10) ## By default, cv is 5
elastic_net_cv_reg.fit(X_train, y_train)

y_pred = elastic_net_cv_reg.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mse)
score = r2_score(y_test, y_pred)
adjusted_score = 1 - (1-score)*(len(y_test) - 1) / (len(y_test) - X_test.shape[1] -1)

print(f"Mean Squared Error: {mse}")
print(f"Mean Absolute Error: {mae}")
print(f"Root Mean Squared Error: {rmse}")
print(f"R2 Score: {score}")
print(f"Adjusted R2 Score: {adjusted_score}")

Mean Squared Error: 133270047.39569493
Mean Absolute Error: 9143.284243933542
Root Mean Squared Error: 11544.264697056064
R2 Score: 0.11678016577627204
Adjusted R2 Score: 0.09510605941495354


In [25]:
poly_reg_lst[0]

{'degree': 2,
 'model': LinearRegression(),
 'x_train': array([[ 1., 24.,  0., ...,  1.,  0.,  0.],
        [ 1., 28.,  1., ...,  0.,  0.,  1.],
        [ 1., 51.,  0., ...,  0.,  0.,  0.],
        ...,
        [ 1., 58.,  0., ...,  0.,  0.,  0.],
        [ 1., 37.,  1., ...,  0.,  0.,  0.],
        [ 1., 55.,  0., ...,  0.,  0.,  0.]], shape=(1003, 45)),
 'x_test': array([[ 1., 45.,  1., ...,  0.,  0.,  0.],
        [ 1., 36.,  1., ...,  1.,  0.,  0.],
        [ 1., 64.,  1., ...,  1.,  0.,  0.],
        ...,
        [ 1., 34.,  0., ...,  0.,  0.,  0.],
        [ 1., 19.,  0., ...,  0.,  0.,  1.],
        [ 1., 41.,  0., ...,  0.,  0.,  1.]], shape=(335, 45)),
 'poly': PolynomialFeatures()}

In [24]:
import pickle
pickle.dump(poly_reg_lst[0]['model'], open(r'D:\Projects\Machine Learning\Supervised\Regression\medical_cost_prediction\models\poly_reg_2.pkl', 'wb'))
pickle.dump(poly_reg_lst[0]['poly'], open(r'D:\Projects\Machine Learning\Supervised\Regression\medical_cost_prediction\models\poly_features.pkl', 'wb'))


In [26]:
sample = pd.DataFrame([{
    "age": 19,
    "sex": 1,       # female, per your notebook
    "bmi": 27.9,
    "children": 0,
    "smoker": 1,
    "northeast": 0,
    "northwest": 0,
    "southeast": 0
}])

sample_poly = poly_reg_lst[0]["poly"].transform(sample)
print(poly_reg_lst[0]["model"].predict(sample_poly)[0])

24494.742653841102
